# 23 - Live models trained on a longer normal run

The first live models were trained on 90 minutes of normal traffic, and their LSTM never fired. Here they are retrained on the first 2 hours of a 3-hour normal run, checked for false alerts on the last hour (never seen in training), and put through the same fault-injection protocol as notebook 17.

In [1]:
import sys
from pathlib import Path

import joblib
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluation.fault_latency import detection_results, false_alerts, load_alerts, load_runs, scenario_summary
from src.streaming.common import LIVE_SERVICES
from src.streaming.offline_replay import replay_live_logs

DOCS, PROCESSED, MODELS = ROOT / "docs", ROOT / "data" / "processed", ROOT / "models"
RUN = ROOT / "data" / "raw" / "live_long"
MODELS_LIVE = ["PCA", "Isolation Forest", "LSTM"]
pd.set_option("display.max_colwidth", 120)

## Data and models

`data/raw/live_long/`: 3 hours of normal traffic (Locust 20 users, 120 SSH logins, no faults), 10:59 PM - 1:59 AM Pakistan time. Split with `scripts/split_live_run.py` on minute boundaries: training before 19:59 UTC (2 hours), held out 19:59 - 20:59 UTC (60 full minutes); the last 15 s are dropped.

The new packs (`models/stream_<service>_long.joblib`, `python -m src.streaming.packs live --dir data/raw/live_long/train --suffix _long`) use the same parser, 1-minute windows, models and thresholds (99th percentile, LSTM k = 9) as the current ones. The detection service loads them with `LIVE_PACK_SUFFIX=_long`; the default is unchanged.

In [2]:
print(f"Run: {(RUN / 'start_time.txt').read_text().strip()} - {(RUN / 'end_time.txt').read_text().strip()}")
pd.DataFrame([{"service": s, "packs": name, "training windows": p["train_windows"], "templates": len(p["event_cols"]),
               "models": ", ".join(p["detectors"])}
              for s in LIVE_SERVICES for name, suffix in [("current (90 min)", ""), ("long (2 h)", "_long")]
              for p in [joblib.load(MODELS / f"stream_{s}{suffix}.joblib")]]).style.hide(axis="index")

Run: 2026-11-25T17:59:13Z - 2026-11-25T20:59:47Z


service,packs,training windows,templates,models
nginx,current (90 min),91,3,"PCA, Isolation Forest, LSTM"
nginx,long (2 h),120,3,"PCA, Isolation Forest, LSTM"
app,current (90 min),91,3,"PCA, Isolation Forest, LSTM"
app,long (2 h),120,2,"PCA, Isolation Forest, LSTM"
postgres,current (90 min),91,9,"PCA, Isolation Forest, LSTM"
postgres,long (2 h),120,9,"PCA, Isolation Forest, LSTM"
ssh,current (90 min),48,2,"PCA, Isolation Forest, LSTM"
ssh,long (2 h),63,2,"PCA, Isolation Forest, LSTM"


## False alerts on the held-out hour

The held-out hour is replayed through the detection service's code (`src/streaming/offline_replay.py`: same parsing, windows, scoring and alert rules; windows close on the log's own clock). Both pack sets are run, so they are compared on the same unseen normal traffic. A New-template warning is counted on every window (as in the fault runs).

In [3]:
rows, heldout_alerts = [], {}
for packs, suffix in [("current", ""), ("long", "_long")]:
    alerts, windows = replay_live_logs(RUN / "heldout", suffix)
    heldout_alerts[packs] = alerts
    for s in LIVE_SERVICES:
        a = alerts[alerts["source"] == s] if len(alerts) else alerts
        model_alerts = a[a["level"] == "Alert"] if len(a) else a
        row = {"packs": packs, "service": s, "windows": windows[s], "hours": 1.0,
               "alerts": len(model_alerts), "new_template_warnings": int((a["level"] == "New template").sum()) if len(a) else 0}
        row.update({m: int(sum(m in f for f in model_alerts["flagged_by"])) if len(model_alerts) else 0 for m in MODELS_LIVE})
        rows.append(row)
heldout = pd.DataFrame(rows)
heldout.to_csv(DOCS / "live_long_false_alerts.csv", index=False)
heldout.style.hide(axis="index")

packs,service,windows,hours,alerts,new_template_warnings,PCA,Isolation Forest,LSTM
current,nginx,60,1.000000,0,0,0,0,0
current,app,60,1.000000,0,0,0,0,0
current,postgres,60,1.000000,0,0,0,0,0
current,ssh,30,1.000000,1,0,1,1,0
long,nginx,60,1.000000,0,0,0,0,0
long,app,60,1.000000,0,0,0,0,0
long,postgres,60,1.000000,1,0,1,0,0
long,ssh,30,1.000000,0,0,0,0,0


In [4]:
pd.concat({p: a[["source", "level", "window", "model", "flagged_by"]] for p, a in heldout_alerts.items() if len(a)}, names=["packs"]) \
    .reset_index(level=0)

,packs,source,level,window,model,flagged_by
0,current,ssh,Alert,20:27-20:28 UTC,PCA,"[PCA, Isolation Forest]"
0,long,postgres,Alert,20:46-20:47 UTC,PCA,[PCA]


## Fault injection with the new packs

`scripts/fault_run_long.sh`: the detection service with `LIVE_PACK_SUFFIX=_long` (every alert written to `data/processed/fault_alerts_long.jsonl`), the live collector, and `scripts/inject_faults.py --runs 5`: database failure, traffic overload and password guessing, 5 runs each, with normal traffic throughout and 7 minutes between faults. Detected: an Alert for an affected service within 3 minutes of the fault start; latency: first such alert - fault start.

In [5]:
faults, start, end = load_runs(DOCS / "fault_runs_long.csv")
alerts = load_alerts(PROCESSED / "fault_alerts_long.jsonl")
print(f"Experiment: {start:%Y-%m-%d %H:%M} - {end:%H:%M} UTC, {len(faults)} runs")
results = detection_results(faults, alerts, within=180)
results.round(1).to_csv(DOCS / "latency_results_long.csv", index=False)
results.drop(columns=["fault_end"]).style.format({"latency_s": "{:.0f}", "latency_any_s": "{:.0f}"}, na_rep="-").hide(axis="index")

Experiment: 2026-11-26 08:52 - 10:57 UTC, 15 runs


/tmp/ipykernel_135357/3529566418.py:5: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  results.round(1).to_csv(DOCS / "latency_results_long.csv", index=False)


run,scenario,fault_start,detected,latency_s,first_alert_service,models_fired,alerts,new_template_warnings,detected_any,latency_any_s,levels
1,db,2026-11-26 08:57:19.675000+00:00,True,41,app,"Isolation Forest, PCA",6,0,True,41,Alert
2,overload,2026-11-26 09:05:19.998000+00:00,True,40,nginx,"Isolation Forest, PCA",6,0,True,40,Alert
3,ssh,2026-11-26 09:13:20.585000+00:00,True,40,ssh,Isolation Forest,1,1,True,40,"Alert, New template"
4,db,2026-11-26 09:21:18.719000+00:00,True,46,app,"Isolation Forest, PCA",6,0,True,46,Alert
5,overload,2026-11-26 09:29:19.028000+00:00,True,41,nginx,"Isolation Forest, PCA",5,0,True,41,Alert
6,ssh,2026-11-26 09:37:19.538000+00:00,True,42,ssh,Isolation Forest,2,0,True,42,Alert
7,db,2026-11-26 09:45:17.683000+00:00,True,46,app,"Isolation Forest, PCA",6,0,True,46,Alert
8,overload,2026-11-26 09:53:18.024000+00:00,True,42,app,"Isolation Forest, PCA",5,0,True,42,Alert
9,ssh,2026-11-26 10:01:18.331000+00:00,True,42,ssh,Isolation Forest,2,0,True,42,Alert
10,db,2026-11-26 10:09:16.433000+00:00,True,45,app,"Isolation Forest, PCA",5,1,True,45,"Alert, New template"


Per scenario, with the number of runs in which each model raised an Alert:

In [6]:
summary = scenario_summary(results)
fired = results.assign(model=results["models_fired"].str.split(", ")).explode("model")
fired = pd.crosstab(fired["scenario"], fired["model"]).reindex(columns=MODELS_LIVE, fill_value=0)
fault_summary = summary.merge(fired.add_suffix(" runs").reset_index(), on="scenario", how="left").fillna(0)
fault_summary.round(1).to_csv(DOCS / "fault_results_long.csv", index=False)
fault_summary.style.format({"detection_rate": "{:.0%}"} | {c: "{:.0f}" for c in fault_summary.columns if c.endswith("_s")}, na_rep="-").hide(axis="index")

scenario,runs,detected,detection_rate,median_latency_s,min_latency_s,max_latency_s,detected_incl_new_template,median_latency_incl_new_template_s,PCA runs,Isolation Forest runs,LSTM runs
db,5,5,100%,46,41,46,5,46,5,5,0
overload,5,5,100%,42,40,45,5,42,5,5,0
ssh,5,5,100%,42,40,109,5,42,0,5,0


False alerts in the normal periods between the faults (same rule as notebook 17):

In [7]:
normal_alerts, normal_hours, fa = false_alerts(faults, alerts, start, end, recovery=120)
print(f"Normal traffic between faults: {normal_hours:.2f} h, Alerts: {int((normal_alerts['level'] == 'Alert').sum())}, "
      f"New-template warnings: {int((normal_alerts['level'] == 'New template').sum())}")
normal_alerts[["alert_time", "source", "level", "flagged_by"]]

Normal traffic between faults: 1.03 h, Alerts: 4, New-template warnings: 0


,alert_time,source,level,flagged_by
54,2026-11-26 10:31:04.961+00:00,postgres,Alert,[PCA]
68,2026-11-26 10:58:04.401+00:00,postgres,Alert,[PCA]
69,2026-11-26 10:58:05.483+00:00,nginx,Alert,"[PCA, Isolation Forest]"
70,2026-11-26 10:58:05.491+00:00,app,Alert,[Isolation Forest]


## Old and new side by side

Old: notebook 17 (current packs, `docs/fault_runs.csv` and `data/processed/fault_alerts.jsonl`).

In [8]:
old_faults, old_start, old_end = load_runs(DOCS / "fault_runs.csv")
old_alerts = load_alerts(PROCESSED / "fault_alerts.jsonl")
old = detection_results(old_faults, old_alerts, within=180)
old_normal, old_hours, _ = false_alerts(old_faults, old_alerts, old_start, old_end, recovery=120)

def models_per_scenario(r):
    m = r.assign(model=r["models_fired"].str.split(", ")).explode("model")
    m = m[m["model"].fillna("") != ""]
    return m.groupby("scenario", sort=False)["model"].agg(lambda s: ", ".join(f"{k} ({v})" for k, v in s.value_counts().items()))

compare = pd.DataFrame({
    "detected (old)": old.groupby("scenario", sort=False)["detected"].sum(),
    "detected (new)": results.groupby("scenario", sort=False)["detected"].sum(),
    "median latency s (old)": old.groupby("scenario", sort=False)["latency_s"].median(),
    "median latency s (new)": results.groupby("scenario", sort=False)["latency_s"].median(),
    "models, runs (old)": models_per_scenario(old),
    "models, runs (new)": models_per_scenario(results),
})
print(f"False Alerts between faults: old {int((old_normal['level'] == 'Alert').sum())} in {old_hours:.2f} h, "
      f"new {int((normal_alerts['level'] == 'Alert').sum())} in {normal_hours:.2f} h")
compare.style.format({c: "{:.1f}" for c in compare.columns if "latency" in c})

False Alerts between faults: old 1 in 1.03 h, new 4 in 1.03 h


,detected (old),detected (new),median latency s (old),median latency s (new),"models, runs (old)","models, runs (new)"
scenario,,,,,,
db,5,5,60.4,45.5,"Isolation Forest (5), PCA (5)","Isolation Forest (5), PCA (5)"
overload,5,5,57.1,42.0,PCA (5),"Isolation Forest (5), PCA (5)"
ssh,5,5,58.1,42.0,"Isolation Forest (5), PCA (1)",Isolation Forest (5)


## Findings

- **Detection: 15 of 15 again**, with the same models: PCA and Isolation Forest for the database failure, Isolation Forest for password guessing. For the traffic overload Isolation Forest now fires as well as PCA (5 of 5 runs, before 0). **The LSTM still never fired.**
- **Latency is lower (median 45.5 / 42.0 / 42.0 s instead of 60.4 / 57.1 / 58.1 s), but not because of the models.** In both experiments an alert comes about 0.6 s after the end of the 1-minute window the fault starts in. These faults started 15-20 s into a minute, the old ones 0-5 s, so their windows closed sooner. One ssh run took 109 s: its first window only gave a New-template warning (at 44 s), the Alert came one window later.
- **False alerts on the held-out hour: 1 with each pack set** (current: ssh, PCA and Isolation Forest; long: postgres, PCA).
- **False alerts between the faults: 4 in 1.03 h** (old: 1). Three of them are the minute in which the background traffic was switched off at the end of the experiment (10:57 UTC, only 13 s of traffic), a known effect; the old experiment ended just before a minute boundary. Without that minute: 1 false alert (postgres, PCA), the same as before.
- **Conclusion:** 2 hours of training (120 windows instead of 91) changes little. The detectors that already worked keep working, Isolation Forest also catches the overload, and the LSTM is still silent. A likely reason (not tested here): each service has only 2-9 event types and the order of events stays normal-looking during these faults, so there is little for it to predict wrongly; the faults show up in event counts, which PCA and Isolation Forest measure. More hours of the same traffic probably won't change that.